# 02_features.ipynb

Préparation des features : détection des vecteurs pré-calculés ou construction depuis `main_dataset.csv`.
- One-hot `squid`
- Moyenne embeddings Word2Vec pour `token_line_m2..token_line_2` (5 × dim)
- Concaténation et sauvegarde dans `data/processed/`

In [1]:
# Imports et chemins
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from gensim.models import Word2Vec

# Détecter la racine du projet de manière robuste (cherche README.md ou dossier data)
def find_project_root(max_up=5):
    p = Path('.').resolve()
    for _ in range(max_up):
        if (p / 'README.md').exists() or (p / 'data').exists() or (p / 'src').exists():
            return p
        p = p.parent
    return Path('.').resolve()

PROJECT_ROOT = find_project_root()
DATA_INPUT = PROJECT_ROOT / 'data' / 'input'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
MODELS_GENSIM = PROJECT_ROOT / 'models' / 'gensim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_INPUT:', DATA_INPUT)
print('MODELS_GENSIM:', MODELS_GENSIM)

PROJECT_ROOT: C:\Projects\SonarQube_FP
DATA_INPUT: C:\Projects\SonarQube_FP\data\input
MODELS_GENSIM: C:\Projects\SonarQube_FP\models\gensim


In [2]:
# Réutiliser utilitaires locaux si présents (ajoute la racine au sys.path)
import sys
from pathlib import Path
import re
# Trouver la racine du projet en montant l'arborescence (cherche 'src' ou 'README.md' ou 'data')
def find_project_root(max_up=6):
    p = Path('.').resolve()
    for _ in range(max_up):
        if (p / 'src').exists() or (p / 'README.md').exists() or (p / 'data').exists():
            return p
        p = p.parent
    return Path('.').resolve()
# Si le notebook est lancé depuis le dossier 'notebooks', utiliser le parent comme racine
cwd = Path('.').resolve()
if cwd.name == 'notebooks' and (cwd.parent / 'src').exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = find_project_root()
print('Using PROJECT_ROOT =', PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# diagnostic: show first sys.path entries
print('sys.path[0:3]=', sys.path[0:3])
try:
    from src.preprocessing import tokenize, get_sentence_embedding, build_feature_matrix
    print('Imported helpers from src.preprocessing')
except Exception as e:
    print('Could not import src.preprocessing helpers:', e)
    # fallback implementations (au cas où)
    def tokenize(text):
        text = (text or '').lower()
        text = re.sub(r'[^a-z0-9\s_]', ' ', text)
        return text.split()
    def get_sentence_embedding(tokens, model, size):
        vectors = [model.wv[t] for t in tokens if t in model.wv]
        if not vectors:
            return np.zeros(size)
        return np.mean(vectors, axis=0)
    def build_feature_matrix(messages, model, size):
        tokenized = [tokenize(m) for m in messages]
        return np.vstack([get_sentence_embedding(t, model, size) for t in tokenized])

Using PROJECT_ROOT = C:\Projects\SonarQube_FP
sys.path[0:3]= ['C:\\Projects\\SonarQube_FP', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\python311.zip', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\DLLs']
Imported helpers from src.preprocessing


In [3]:
# Détection de features pré-calculés (vérifie `data/input` puis `data/processed`)
# Diagnostic: afficher chemins absolus et existence pour debug
print('Diagnostic: cwd=', Path('.').resolve())
cands = [Path('.').resolve(), Path('..').resolve(), Path('..').resolve().parent]
for c in cands:
    print('Checking candidate root:', c)
    print(' -', c / 'data' / 'input', 'exists=', (c / 'data' / 'input').exists())
    if (c / 'data' / 'input').exists():
        print('   contents:', [p.name for p in (c / 'data' / 'input').glob('*')])
    print(' -', c / 'data' / 'processed', 'exists=', (c / 'data' / 'processed').exists())
    if (c / 'data' / 'processed').exists():
        print('   contents:', [p.name for p in (c / 'data' / 'processed').glob('*')])
# Ensure DATA_INPUT and PROCESSED_DIR point at PROJECT_ROOT's data folders (in case earlier cells weren't re-run)
DATA_INPUT = PROJECT_ROOT / 'data' / 'input'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print('Resolved DATA_INPUT ->', DATA_INPUT)
print('Resolved PROCESSED_DIR ->', PROCESSED_DIR)
input_files = list(DATA_INPUT.glob('*')) if DATA_INPUT.exists() else []
processed_files = list(PROCESSED_DIR.glob('*')) if PROCESSED_DIR.exists() else []
print('Fichiers dans data/input:', [p.name for p in input_files])
print('Fichiers dans data/processed:', [p.name for p in processed_files])
has_features = False
# helper pour trouver un unique X (x.npy / x.csv / x.pkl) dans une liste de fichiers
def _find_single_x(file_list):
    return next((p for p in file_list if p.suffix.lower() in ('.npy', '.pkl', '.joblib', '.npz', '.csv') and p.stem.lower() == 'x'), None)
# recherche dans data/input puis data/processed
single_X = _find_single_x(input_files) or _find_single_x(processed_files)
# helper pour chercher X_train / X_test dans un dossier donné
def _find_split(folder, base):
    for name in (f'{base}.csv', f'{base}.npy', f'{base}.pkl', f'{base}.joblib'):
        p = folder / name
        if p.exists():
            return p
    return None
x_train_path = _find_split(DATA_INPUT, 'x_train') or _find_split(DATA_INPUT, 'X_train') or _find_split(PROCESSED_DIR, 'X_train') or _find_split(PROCESSED_DIR, 'x_train')
x_test_path = _find_split(DATA_INPUT, 'x_test') or _find_split(DATA_INPUT, 'X_test') or _find_split(PROCESSED_DIR, 'X_test') or _find_split(PROCESSED_DIR, 'x_test')
if single_X is not None:
    print('Found single X file:', single_X)
    if single_X.suffix.lower() == '.npy':
        X = np.load(single_X)
    elif single_X.suffix.lower() in ('.pkl', '.joblib'):
        X = joblib.load(single_X)
    elif single_X.suffix.lower() == '.csv':
        X = pd.read_csv(single_X, header=None).values
    has_features = True
    # attempt to load y if present (search both input and processed)
    y_path = (DATA_INPUT / 'y_train.csv') if (DATA_INPUT / 'y_train.csv').exists() else ((PROCESSED_DIR / 'y_train.npy') if (PROCESSED_DIR / 'y_train.npy').exists() else (PROCESSED_DIR / 'y_train.csv' if (PROCESSED_DIR / 'y_train.csv').exists() else None))
    if y_path is not None:
        if y_path.suffix.lower() in ('.npy',):
            y = np.load(y_path)
        else:
            y = pd.read_csv(y_path, header=None).squeeze().values
    print('Loaded single X shape:', getattr(X, 'shape', None))
elif x_train_path is not None or x_test_path is not None:
    print('Found X_train/X_test — loading')
    def _load_any(p):
        if p is None:
            return None
        s = p.suffix.lower()
        if s == '.npy':
            return np.load(p)
        if s == '.csv':
            return np.loadtxt(p, delimiter=',')
        return joblib.load(p)
    X_train = _load_any(x_train_path)
    X_test = _load_any(x_test_path)
    has_features = True
    # load y_train/y_test if present (check input then processed)
    def _find_y(name):
        for folder in (DATA_INPUT, PROCESSED_DIR):
            p = folder / name
            if p.exists():
                return p
        return None
    y_train_path = _find_y('y_train.csv') or _find_y('y_train.npy')
    y_test_path = _find_y('y_test.csv') or _find_y('y_test.npy')
    y_train = pd.read_csv(y_train_path, header=None).squeeze().values if (y_train_path and y_train_path.suffix.lower() == '.csv') else (np.load(y_train_path) if y_train_path and y_train_path.suffix.lower() == '.npy' else (joblib.load(y_train_path) if y_train_path else None))
    y_test = pd.read_csv(y_test_path, header=None).squeeze().values if (y_test_path and y_test_path.suffix.lower() == '.csv') else (np.load(y_test_path) if y_test_path and y_test_path.suffix.lower() == '.npy' else (joblib.load(y_test_path) if y_test_path else None))
    print('Loaded X_train shape:', getattr(X_train, 'shape', None), 'X_test shape:', getattr(X_test, 'shape', None))
else:
    print('No precomputed X found — will try to build from dataset')

Diagnostic: cwd= C:\Projects\SonarQube_FP\notebooks
Checking candidate root: C:\Projects\SonarQube_FP\notebooks
 - C:\Projects\SonarQube_FP\notebooks\data\input exists= False
 - C:\Projects\SonarQube_FP\notebooks\data\processed exists= False
Checking candidate root: C:\Projects\SonarQube_FP
 - C:\Projects\SonarQube_FP\data\input exists= True
   contents: ['x_test.csv', 'x_train.csv', 'y_test.csv', 'y_train.csv']
 - C:\Projects\SonarQube_FP\data\processed exists= True
   contents: []
Checking candidate root: C:\Projects
 - C:\Projects\data\input exists= False
 - C:\Projects\data\processed exists= False
Resolved DATA_INPUT -> C:\Projects\SonarQube_FP\data\input
Resolved PROCESSED_DIR -> C:\Projects\SonarQube_FP\data\processed
Fichiers dans data/input: ['x_test.csv', 'x_train.csv', 'y_test.csv', 'y_train.csv']
Fichiers dans data/processed: []
Found X_train/X_test — loading
Loaded X_train shape: (39263, 480) X_test shape: (4433, 480)


In [4]:
# Si pas de features, on cherche main_dataset.csv — SKIP si features déjà présentes
if not has_features:
    MAIN_CSV_CANDIDATES = [Path('data') / 'raw' / 'main_dataset.csv', Path('main_dataset.csv'), Path('data') / 'main_dataset.csv']
    main_csv = next((p for p in MAIN_CSV_CANDIDATES if p.exists()), None)
    if main_csv is None:
        print('main_dataset.csv introuvable. Pour construire X, placez le fichier CSV à lune des locations suivantes:')
        for p in MAIN_CSV_CANDIDATES:
            print(' -', p)
        raise FileNotFoundError('main_dataset.csv manquant — impossible de construire les features sans dataset')
    print('Found main dataset at', main_csv)
    df = pd.read_csv(main_csv, low_memory=False)
    print('Loaded dataframe shape:', df.shape)
    print('Columns:', list(df.columns))
else:
    print('Precomputed features found — skipping main dataset load')

Precomputed features found — skipping main dataset load


In [5]:
# Charger le modèle word2vec (seulement si on doit construire les embeddings)
if not has_features:
    model_files = list(MODELS_GENSIM.glob('*.model'))
    if not model_files:
        raise FileNotFoundError('Aucun modèle gensim trouvé dans models/gensim — placez le fichier .model')
    model_path = model_files[0]
    print('Loading gensim model:', model_path)
    model = Word2Vec.load(str(model_path))
    vec_size = model.vector_size
    print('Word2Vec vector size:', vec_size)
else:
    print('Skipping Word2Vec load because precomputed features are available')

Skipping Word2Vec load because precomputed features are available


In [6]:
# Construire features depuis le dataframe seulement si nécessaire
if not has_features:
    # One-hot encoding de `squid` si présent
    if 'squid' not in df.columns:
        raise KeyError('La colonne "squid" est absente du dataset — impossible de faire one-hot encoding')
    squid_dummies = pd.get_dummies(df['squid'], prefix='squid')
    print('One-hot squid shape:', squid_dummies.shape)
    # Embeddings pour token_line_*
    token_cols = ['token_line_m2', 'token_line_m1', 'token_line_0', 'token_line_1', 'token_line_2']
    emb_list = []
    for col in token_cols:
        if col not in df.columns:
            print(f'Colonne {col} absente — remplissage par zéros')
            emb = np.zeros((len(df), vec_size))
        else:
            msgs = df[col].fillna('').astype(str).tolist()
            emb = build_feature_matrix(msgs, model, vec_size)
        print(f'Embedding {col} shape:', emb.shape)
        emb_list.append(emb)
    # Concaténation embeddings (N x 5*vec_size)
    X_emb = np.hstack(emb_list) if emb_list else np.empty((len(df), 0))
    print('Concatenated embeddings shape:', X_emb.shape)
    # Concaténation final: one-hot squid + embeddings
    X = np.hstack([squid_dummies.values, X_emb])
    print('Final X shape:', X.shape)
    # Labels
    if 'cls' not in df.columns:
        raise KeyError('La colonne "cls" (labels) est absente du dataset')
    y = df['cls'].values
    print('y shape:', y.shape, 'unique counts:', dict(pd.Series(y).value_counts()))
else:
    print('Precomputed features detected — skipping dataframe-based feature construction')
    # Si des splits pré-calculés sont chargés, réutiliser X_train / X_test / y_train / y_test
    if 'X_train' in globals():
        X = X_train
        y = y_train if 'y_train' in globals() else None
        print('Using X_train shape:', getattr(X, 'shape', None))
    elif 'X' in globals():
        print('Using precomputed single X shape:', getattr(X, 'shape', None))
    else:
        print('No X variables found in globals; nothing to build')

Precomputed features detected — skipping dataframe-based feature construction
Using X_train shape: (39263, 480)


In [8]:
# Sauvegarder X et y pour réutilisation (supporte les splits préchargés)
if 'X_train' in globals() or 'X' in globals():
    if 'X_train' in globals():
        joblib.dump(X_train, PROCESSED_DIR / 'X_train.pkl')
        np.save(PROCESSED_DIR / 'X_train.npy', X_train)
    if 'X_test' in globals() and X_test is not None:
        joblib.dump(X_test, PROCESSED_DIR / 'X_test.pkl')
        np.save(PROCESSED_DIR / 'X_test.npy', X_test)
    if 'X' in globals():
        joblib.dump(X, PROCESSED_DIR / 'X.pkl')
        np.save(PROCESSED_DIR / 'X.npy', X)
if 'y_train' in globals() or 'y' in globals():
    if 'y_train' in globals():
        joblib.dump(y_train, PROCESSED_DIR / 'y_train.pkl')
        np.save(PROCESSED_DIR / 'y_train.npy', y_train)
    if 'y_test' in globals() and y_test is not None:
        joblib.dump(y_test, PROCESSED_DIR / 'y_test.pkl')
        np.save(PROCESSED_DIR / 'y_test.npy', y_test)
    if 'y' in globals():
        joblib.dump(y, PROCESSED_DIR / 'y.pkl')
        np.save(PROCESSED_DIR / 'y.npy', y)
print('Saved available X/y artifacts to', PROCESSED_DIR)

Saved available X/y artifacts to C:\Projects\SonarQube_FP\data\processed


## Notes:
- Si `data/input` contenait déjà des vecteurs, le notebook tente de les charger prioritairement.
- Si `main_dataset.csv` est absent, placez le dataset principal dans `data/raw/main_dataset.csv` ou `main_dataset.csv` à la racine.
- Conserver `random_state=42` pour la reproductibilité lors des étapes ultérieures.